In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
df = pd.read_csv("Housing.csv")
df.head()

In [ ]:
print('Number of records:', df.shape[0])
print('Number of columns:', df.shape[1])

In [ ]:
df.info()

In [ ]:
for i in df.columns:
    print(df[i].value_counts())

In [ ]:
df['mainroad'] = df['mainroad'].map({'yes': 1, 'no': 0})
df['guestroom'] = df['guestroom'].map({'yes': 1, 'no': 0})
df['basement'] = df['basement'].map({'yes': 1, 'no': 0})
df['hotwaterheating'] = df['hotwaterheating'].map({'yes': 1, 'no': 0})
df['airconditioning'] = df['airconditioning'].map({'yes': 1, 'no': 0})
df['prefarea'] = df['prefarea'].map({'yes': 1, 'no': 0})
df['furnishingstatus'] = df['furnishingstatus'].map({'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0})

In [ ]:
df

In [ ]:
fig, ax = plt.subplots(figsize=(19, 10), dpi=50)
df.hist(ax=ax, layout=(3, 5), alpha=0.5)

In [ ]:
sns.heatmap(df.corr(numeric_only=True), cmap='RdYlBu')

In [ ]:
df.isnull().sum()

In [ ]:
from sklearn.preprocessing import StandardScaler

def normalize(X):
    print("Mean and Standard Deviation Before")
    print(X.mean(axis=0), X.std(axis=0))

    sc = StandardScaler()
    XScaled = sc.fit_transform(X)

    print("Mean and Standard Deviation After")
    print(XScaled.mean(axis=0).round(4), XScaled.std(axis=0))
    return XScaled

In [ ]:
from sklearn.model_selection import train_test_split

def splitTrainTest(X, Y, seed, splitRatio):
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=splitRatio, random_state=seed)
    print("Dimensions of Original Data:")
    print("Size(X):", X.shape, "; Size(Y)", Y.shape)
    print("Dimensions of Training Data:")
    print("Size(X_train):", X_train.shape, "; Size(Y_train)", Y_train.shape)
    print("Dimensions of Test Data:")
    print("Size(X_test):", X_test.shape, "; Size(Y_test)", Y_test.shape)
    return X_train, X_test, Y_train, Y_test

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn import metrics

features = df.columns[1:]  # All features except 'price'

X = df[features].values
Y = df['price'].values

print("*************Normalization/Standardization*************")
XScaled = normalize(X)

In [ ]:
def ANNRegression(XScaled, Y):
    print("*************Train-Test Split*************")
    X_train, X_test, Y_train, Y_test = splitTrainTest(XScaled, Y, seed=2, splitRatio=0.2)

    print("\n*************Learning/Fitting the ANN Regression Model*************")
   
    regModel = MLPRegressor(hidden_layer_sizes=(256, 128, 64, 32),
                            activation='relu',
                            solver='sgd',
                            max_iter=1000,
                            random_state=42)
    regModel.fit(X_train, Y_train)

    print("Number of Hidden Layers:", regModel.n_layers_ - 2)
    print("Number of Iterations Run:", regModel.n_iter_)

    print("\n*************Evaluating Performance on Test Partition*************")
    Y_pred = regModel.predict(X_test)
    print('MAE:', metrics.mean_absolute_error(Y_test, Y_pred))
    print('MSE:', metrics.mean_squared_error(Y_test, Y_pred))
    print('RMSE:', np.sqrt(metrics.mean_squared_error(Y_test, Y_pred)))
    print('R2 Score:', metrics.r2_score(Y_test, Y_pred))

    return regModel

In [ ]:
ANNRegression(XScaled, Y)

In [ ]:
from sklearn.model_selection import KFold

print("*************K-Fold Cross Validation (ANN Regression)*************")

kf = KFold(n_splits=5, shuffle=True, random_state=2)

mse_list = []
r2_list = []

for train_index, test_index in kf.split(XScaled):
    X_train_k = XScaled[train_index]
    X_test_k = XScaled[test_index]
    Y_train_k = Y[train_index]
    Y_test_k = Y[test_index]

    model_k = MLPRegressor(hidden_layer_sizes=(256, 128, 64, 32),
                           activation='relu',
                           solver='sgd',
                           max_iter=1000,
                           random_state=42)
    model_k.fit(X_train_k, Y_train_k)

    Y_pred_k = model_k.predict(X_test_k)

    mse_list.append(metrics.mean_squared_error(Y_test_k, Y_pred_k))
    r2_list.append(metrics.r2_score(Y_test_k, Y_pred_k))

print("Average K-Fold MSE:", np.mean(mse_list))
print("Average K-Fold R2 :", np.mean(r2_list))

In [ ]:
iris_df = sns.load_dataset('iris')
iris_df.head()

In [ ]:
print('Number of records:', iris_df.shape[0])
print('Number of columns:', iris_df.shape[1])

In [ ]:
iris_df.info()

In [ ]:
print(iris_df['species'].value_counts())

In [ ]:
iris_df.isnull().sum()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), dpi=80)
iris_df.hist(ax=ax, layout=(2, 2), alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(iris_df, hue='species')

In [ ]:
# Retrieve X (all 4 attributes) and Y (all 3 class labels)
X_iris = iris_df.drop('species', axis=1).values
Y_iris = iris_df['species'].values

print("Shape of X:", X_iris.shape)
print("Shape of Y:", Y_iris.shape)
print("Classes:", np.unique(Y_iris))

In [ ]:
print("*************Normalization/Standardization*************")
XScaled_iris = normalize(X_iris)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from mlxtend.plotting import plot_confusion_matrix

print("*************K-Fold Cross Validation (ANN Classification)*************")


kf = KFold(n_splits=5, shuffle=True, random_state=2)

train_acc_list = []
test_acc_list = []
fold = 1

for train_index, test_index in kf.split(XScaled_iris):

    print(f"\n===== Fold {fold} =====")

    X_train, X_test = XScaled_iris[train_index], XScaled_iris[test_index]
    Y_train, Y_test = Y_iris[train_index], Y_iris[test_index]

    # Create and train model
    clsModel = MLPClassifier(hidden_layer_sizes=(34, 22),
                             activation='relu',
                             solver='sgd',
                             max_iter=1000,
                             random_state=42)
    clsModel.fit(X_train, Y_train)

    # Predictions
    Y_testPred = clsModel.predict(X_test)
    Y_trainPred = clsModel.predict(X_train)

    # Accuracy
    train_acc = accuracy_score(Y_train, Y_trainPred)
    test_acc = accuracy_score(Y_test, Y_testPred)
    train_acc_list.append(train_acc)
    test_acc_list.append(test_acc)

    print(f"TRAIN Accuracy: {train_acc:.4f}")
    print(f"TEST  Accuracy: {test_acc:.4f}")

    # Confusion Matrix
    cm = confusion_matrix(Y_test, Y_testPred)
    print("\nConfusion Matrix:")
    print(cm)
    fig, ax = plot_confusion_matrix(conf_mat=cm,
                                    class_names=np.unique(Y_iris),
                                    show_normed=False,
                                    colorbar=True)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.show()

    # Classification Report
    print("\nClassification Report (Precision, Recall, F1-Score):")
    print(classification_report(Y_test, Y_testPred))

    fold += 1

In [ ]:
print("*************Average Accuracy Across All Folds*************")
print("Average TRAIN Accuracy:", round(np.mean(train_acc_list), 4))
print("Average TEST  Accuracy:", round(np.mean(test_acc_list), 4))